In [ ]:
import pandas as pd
import os
from datetime import datetime

left_data = pd.DataFrame({"id": [1, 1, 2], "value": [10, 11, 20]})
right_data = pd.DataFrame({"id": [1, 1, 2], "name": ["A", "A_dup", "B"]})

merged_bug = left_data.merge(right_data, on="id", how="left")
print(f"ПРОБЛЕМА: Было {len(left_data)} строк -> Стало {len(merged_bug)}")
print(merged_bug)

right_fixed = right_data.drop_duplicates(subset=['id'])
merged_fixed = left_data.merge(right_fixed, on="id", how="left")
print(f"\nИСПРАВЛЕНО: Теперь строк {len(merged_fixed)}")
print(merged_fixed)

ПРОБЛЕМА: Было 3 строк -> Стало 5
   id  value   name
0   1     10      A
1   1     10  A_dup
2   1     11      A
3   1     11  A_dup
4   2     20      B

ИСПРАВЛЕНО: Теперь строк 3
   id  value name
0   1     10    A
1   1     11    A
2   2     20    B


In [4]:
ref_dir = "../data/reference"
os.makedirs(ref_dir, exist_ok=True)

holiday_ref = pd.DataFrame({
    'category': ['Public', 'Optional', 'Observance'],
    'is_work_free': [1, 0, 0],              
    'importance_score': [3, 2, 1]           
})

ref_path = os.path.join(ref_dir, "holiday_types.csv")
holiday_ref.to_csv(ref_path, index=False)
print(f"Справочник сохранен в: {ref_path}")

Справочник сохранен в: ../data/reference\holiday_types.csv


In [ ]:
norm_file = "../data/normalized/variant_20/2026-03-19_11-39-15_holidays_germany.csv" 
df_norm = pd.read_csv(norm_file)

df_ref = pd.read_csv("../data/reference/holiday_types.csv")

df_enriched = df_norm.merge(df_ref, on='category', how='left', validate='many_to_one')

print(f"Данные успешно обогащены. Строк: {len(df_enriched)}")
print(df_enriched.head(3))

Данные успешно обогащены. Строк: 20
  holiday_date                 local_name                   eng_name category  \
0   2025-01-01                    Neujahr             New Year's Day   Public   
1   2025-01-06        Heilige Drei Könige                   Epiphany   Public   
2   2025-03-08  Internationaler Frauentag  International Women's Day   Public   

   is_work_free  importance_score  
0             1                 3  
1             1                 3  
2             1                 3  


In [ ]:
df_enriched['holiday_date'] = pd.to_datetime(df_enriched['holiday_date'])

df_enriched['month'] = df_enriched['holiday_date'].dt.to_period('M').dt.to_timestamp()

mart = df_enriched.groupby('month').agg(
    total_holidays=('local_name', 'count'),      # KPI 1: Сколько всего событий в месяце
    public_holidays=('is_work_free', 'sum'),     # KPI 2: Сколько из них реальных выходных
    avg_importance=('importance_score', 'mean')  # KPI 3: Насколько "важный" месяц
).reset_index()

mart['rolling_avg_3m'] = mart['total_holidays'].rolling(window=3, min_periods=1).mean()

mart_folder = "../data/mart/variant_20"
os.makedirs(mart_folder, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
mart_path = os.path.join(mart_folder, f"mart_monthly_{timestamp}.csv")

mart.to_csv(mart_path, index=False)

print(f"Витрина успешно создана и сохранена:\n{mart_path}")
print("\nФинальная витрина (head):")
print(mart.head())

Витрина успешно создана и сохранена:
../data/mart/variant_20\mart_monthly_2026-03-25_11-10-01.csv

Финальная витрина (head):
       month  total_holidays  public_holidays  avg_importance  rolling_avg_3m
0 2025-01-01               2                2             3.0        2.000000
1 2025-03-01               1                1             3.0        1.500000
2 2025-04-01               3                3             3.0        2.000000
3 2025-05-01               3                3             3.0        2.333333
4 2025-06-01               3                3             3.0        3.000000
